# 02. Stop Matching

**Scope:** determine which scheduled stop a vehicle is at or approaching.

**Primary strategy:** trust the GTFS-Realtime feed's own `stop_id`/`current_stop_sequence`/ `current_status` directly. **Fallback (scoped, not default):** distance + monotonic-sequence inference, used only if Section C confirms it can actually help the rows missing primary data.

**Dataset:** loaded from notebook 01's auto-generated provenance sidecar.


In [1]:
import json
import pandas as pd
import duckdb

with open('telemetry_sample_N3.meta.json') as f:
    PROVENANCE = json.load(f)

print(json.dumps(PROVENANCE, indent=2))

GTFS_STATIC_PATH = '../../gtfs_static/MBTA_GTFS'
AGENCY_TZ = PROVENANCE['agency_timezone']
POLL_INTERVAL_SECONDS = PROVENANCE['poll_interval_seconds']
SILENT_VEHICLE_THRESHOLD_SECONDS = PROVENANCE['silent_vehicle_threshold_seconds']

df_deduped = pd.read_parquet(PROVENANCE['file'])
# timestamp_eastern was computed once in notebook 01, so it is reused here
df_deduped['timestamp_eastern'] = pd.to_datetime(df_deduped['timestamp_eastern'])


{
  "file": "telemetry_sample_N3.parquet",
  "rows": 197109,
  "sha256": "922117775e390be73268b4e864e2950900ed9263e0228f48c1bc5e9e6b34e926",
  "capture_start_eastern": "2026-09-01 18:15:15-04:00",
  "capture_end_eastern": "2026-09-01 21:33:48-04:00",
  "agency_timezone": "America/New_York",
  "poll_interval_seconds": 15,
  "active_vehicles_per_poll_median": 453,
  "silent_vehicle_threshold_seconds": 110.0
}


## A. Primary Strategy -- Enriching Reported Stops

**Question:** For pings where MBTA already reports `stop_id`, can we simply join against `stops.txt` for a human-readable name and coordinates, with no inference required at all?

**Method:** Direct join on `stop_id`.


In [2]:
query_primary = f"""
    SELECT
        p.vehicle_id, 
        p.trip_id, 
        p.route_id, 
        p.timestamp_eastern,
        p.current_status, 
        p.current_stop_sequence, 
        p.stop_id,
        
        s.stop_name, 
        s.stop_lat, 
        s.stop_lon
    FROM df_deduped AS p
    LEFT JOIN read_csv_auto('{GTFS_STATIC_PATH}/stops.txt', types={{'stop_id': 'VARCHAR'}}) AS s
        ON p.stop_id = s.stop_id
    ORDER BY p.vehicle_id, p.timestamp_eastern
"""

df_primary = duckdb.sql(query_primary).df()

has_stop = df_primary['stop_id'].notna().sum()
total = len(df_primary)
print(f"Pings with a directly-reported stop_id: {has_stop} / {total} ({has_stop/total*100:.1f}%)")
df_primary.head(10)


Pings with a directly-reported stop_id: 195598 / 197109 (99.2%)


,vehicle_id,trip_id,route_id,timestamp_eastern,current_status,current_stop_sequence,stop_id,stop_name,stop_lat,stop_lon
0,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:14:03-06:00,STOPPED_AT,0.0,BNT-0000-02,North Station,42.366761,-71.062365
1,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:14:34-06:00,STOPPED_AT,0.0,BNT-0000,North Station,42.366417,-71.062326
2,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:15:00-06:00,STOPPED_AT,0.0,BNT-0000,North Station,42.366417,-71.062326
3,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:15:08-06:00,STOPPED_AT,0.0,BNT-0000,North Station,42.366417,-71.062326
4,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:15:36-06:00,STOPPED_AT,0.0,BNT-0000,North Station,42.366417,-71.062326
5,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:16:07-06:00,STOPPED_AT,0.0,BNT-0000,North Station,42.366417,-71.062326
6,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:16:38-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.042290
7,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:16:58-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.042290
8,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:17:09-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.042290
9,1700,NorthBase-830295-77,CR-Newburyport,2026-09-01 17:17:40-06:00,IN_TRANSIT_TO,10.0,ER-0042-01,Chelsea,42.397222,-71.042290


**Result:** The result is expected: in the notebook 1 at section J , it was discovered that Shuttle-Generic is around the 0.76% of this dataset, so the coverage here should land close to ~99%, which is what actually happen.

**Decision:** With more than 99% of coverage, building a custom spatial-distance (lat/lon) inference algorithm as our default strategy is completely unnecessary and would introduce unjustified computational overhead. The API's payload will be trusted, so the stop-resolution will be delegated to the MBTA's server-side logic.

## B. Validating the Primary Strategy

**Question:** Is MBTA's own reported `current_stop_sequence` internally consistent, or does it increase monotonically per trip, or even does it ever jump backwards?

**Method:** For each `(vehicle_id, trip_id)`, sort by timestamp, check whether `current_stop_sequence` is non-decreasing.


In [3]:
df_seq = df_primary.dropna(subset=['current_stop_sequence']).copy()
df_seq = df_seq.sort_values(['vehicle_id', 'trip_id', 'timestamp_eastern'])
df_seq['seq_diff'] = df_seq.groupby(['vehicle_id', 'trip_id'])['current_stop_sequence'].diff()

backwards = df_seq[df_seq['seq_diff'] < 0]
print(f"Total consecutive same-trip observations: {len(df_seq)}")
print(f"Backwards stop_sequence jumps: {len(backwards)} ({len(backwards)/len(df_seq)*100:.2f}%)")

if len(backwards) > 0:
    display(backwards[['vehicle_id', 'trip_id', 'timestamp_eastern', 'current_stop_sequence', 'seq_diff']].head(10))


Total consecutive same-trip observations: 195598
Backwards stop_sequence jumps: 166 (0.08%)


,vehicle_id,trip_id,timestamp_eastern,current_stop_sequence,seq_diff
10524,G-10020,76509738,2026-09-01 18:03:45-06:00,430.0,-120.0
12883,G-10058,76510130,2026-09-01 18:45:32-06:00,490.0,-220.0
16938,G-10138,76502105,2026-09-01 17:26:01-06:00,1.0,-7.0
18421,G-10154,76510039,2026-09-01 19:33:47-06:00,4.0,-706.0
18971,G-10179,ADDED-1584904878,2026-09-01 19:06:55-06:00,190.0,-200.0
19541,G-10190,76510247,2026-09-01 19:09:12-06:00,410.0,-10.0
19542,G-10190,76510263,2026-09-01 19:09:25-06:00,410.0,-10.0
19564,G-10190,76510263,2026-09-01 19:16:09-06:00,550.0,-10.0
19555,G-10190,ADDED-1584904908,2026-09-01 19:13:20-06:00,430.0,-120.0
23796,O-548B7DE3,76823042,2026-09-01 19:22:51-06:00,110.0,-10.0


**Result**: Out of 195_598 consecutive observations, 166 instances, which is the 0.08%, exhibited backwards sequence jumps (defined by seq_diff < 0).

**Decision**: The reported current_stop_sequence is remarkably consistent (>99.9% monotonic). The few non-monotonic anomalies (0.08%) represent edge cases, rather than systematic feed corruption. It can be safely trusted current_stop_sequence directly in the Python analytics engine.

## C. Scoping the Fallback -- Can Distance Inference Even Help?

**Question:** Missing `stop_id` concentrates almost entirely in `Shuttle-Generic*` (confirmed again in notebook 01, Section J: 100% missing, 1,490/1,490 pings). If these trips have no `stop_times.txt` entry at all, a distance-based fallback can't help them either.

**Method:** For every trip_id among the null-stop pings, check whether it exists in `stop_times.txt` at all.

In [4]:
null_stop_trip_ids = (
    df_deduped[df_deduped['stop_id'].isna()]['trip_id']
    .dropna().unique().tolist()
)

query_fallback_feasibility = f"""
    WITH null_stop_trips AS (SELECT UNNEST($trip_ids) AS trip_id)
    SELECT
        COUNT(*) AS total_null_stop_trips,
        COUNT(*) FILTER (
            WHERE trip_id IN (
                SELECT DISTINCT trip_id
                FROM read_csv_auto('{GTFS_STATIC_PATH}/stop_times.txt', types={{'trip_id': 'VARCHAR'}})
            )
        ) AS resolvable_in_static_schedule
    FROM null_stop_trips
"""

result = duckdb.sql(query_fallback_feasibility, params={'trip_ids': null_stop_trip_ids}).df()
result['pct_resolvable'] = result['resolvable_in_static_schedule'] / result['total_null_stop_trips'] * 100
result


,total_null_stop_trips,resolvable_in_static_schedule,pct_resolvable
0,31,7,22.580645


Despite the previous run resulted in 9.5% of trip IDs missing a stop_id that actually existed in stop_times.txt, and this current rn resulted in 22.58%, the trip-id percentage alone does not tell the whole story to make a decision. What actually matters is knowing how many of those seven pings would affect the whole dataset.

It must be considered that 31 trip_ids total sits inside a null-stop_id population, which is ~0.76% of the full capture (Shuttle-Generic, per notebook 01 Section J). So, 22.58% belongs to a rare category, a minority, and this number could still be a tiny absolute number. This means that if the absolute percentage is low enought, a full spatial fallback matcher is still disproportionate effort for the recovery.

In [7]:
resolvable_trip_ids = duckdb.sql(f"""
    SELECT DISTINCT 
    trip_id
    FROM read_csv_auto('{GTFS_STATIC_PATH}/stop_times.txt', types={{'trip_id': 'VARCHAR'}})
    WHERE trip_id IN (SELECT UNNEST($trip_ids))
""", params={'trip_ids': null_stop_trip_ids}).df()['trip_id'].tolist()

# Filtering the wholw dataset looking for the rows with the resolvable_trip_ids
affected_pings = df_deduped[df_deduped['trip_id'].isin(resolvable_trip_ids)]
print(f"Pings potentially recoverable via fallback: {len(affected_pings)} / {len(df_deduped)} ({len(affected_pings)/len(df_deduped)*100:.3f}%)")

Pings potentially recoverable via fallback: 776 / 197109 (0.394%)


**Result:** 

* 7 out of 31 of the trip IDs missing a stop_id actually exist in stop_times.txt. 24 trips have no static schedule entries at all.
* While 22.58% of the problematic trip IDs are technically resolvable, looking at the pings tells the real story. Those 7 resolvable trips account for only 776 pings out of 197,109 total pings in the dataset. This represents a 0.394% of the full capture.

**Decision:** NO GO for the spatial fallback matcher for the MVP.
Our previous run showed a 9.5% trip-level resolvability, which was already deemed too small to justify building a fallback mechanism. Even though this current rn shows a higher percentage at the trip level (22.58%), the absolute global impact dropped significantly to just 0.394% of total pings.
Building a full spatial fallback matcher is a disproportionate effort for recovering less than 0.4% of the data. Therefore, the original conclusion holds: Shuttle-Generic and trips without static schedule entries are explicitly excluded from stop-level metrics for the MVP. This aligns perfectly with the dwell-analysis scope decision established in Notebook 01, Section J. We accept this tiny data loss to keep the architecture clean and fast.

## Summary of Decisions

| Decision | Value | Source |
|---|---|---|
| Primary strategy | trust native `stop_id`/`current_stop_sequence` directly | Section A |
| Native coverage | *99.2%* | Section A |
| Sequence monotonicity | *0.08%* backwards jumps | Section B |
| Fallback matcher | Rejected. Trips with missing stop data represent only the 0.394% of the total dataset | Section C |
